# 05 — Churn Prediction Model
**Goal**: Train, compare, and explain three models:
1. **Logistic Regression** — interpretable baseline, shows evaluators you understand model hierarchy
2. **XGBoost** — primary model, best performance, produces SHAP explanations
3. **Cox Proportional Hazard** — econometric model, outputs *time-to-churn* probability instead of binary label

**Train/Val/Test split** (strict time-based, no data leakage):
- Train  : members whose prediction window falls before 2016-07
- Val    : 2016-07 to 2016-12
- Test   : held out — 2017 onward (forward window post-cutoff)

**Input** : `data/processed/04_features.csv`
**Output** : `models/churn_model.pkl`, `models/scaler.pkl`, `data/processed/05_predictions.csv`

## 0. Imports & Config

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path
import os, warnings, joblib
warnings.filterwarnings('ignore')

# Modelling
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    classification_report, confusion_matrix,
    RocCurveDisplay, PrecisionRecallDisplay
)
from sklearn.pipeline import Pipeline
import xgboost as xgb
import shap

# Survival analysis
try:
    from lifelines import CoxPHFitter
    from lifelines.utils import concordance_index
    LIFELINES_AVAILABLE = True
    print('lifelines available — Cox model will run.')
except ImportError:
    LIFELINES_AVAILABLE = False
    print('lifelines not installed. Run: pip install lifelines --quiet')
    print('Cox model will be skipped — XGBoost + Logistic still run.')

PROC  = Path('../data/processed')
FIG   = Path('../reports/figures')
MODEL = Path('../models')
os.makedirs(FIG,   exist_ok=True)
os.makedirs(MODEL, exist_ok=True)

KEY = 'loyalty_number'
TARGET = 'churned'

CHURN_COLOR  = '#C0392B'
RETAIN_COLOR = '#2471A3'
sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

print('Setup complete.')

---
## 1. Load Features

In [ ]:
df = pd.read_csv(PROC / '04_features.csv')
print(f'Feature table: {df.shape}')
print(f'Churn rate   : {df[TARGET].mean():.1%}')
display(df.head(3))

---
## 2. Feature Selection & Train / Validation Split
We use **all members** for training — the temporal split is already baked in by the prediction cutoff in notebook 04. Here we split by member index for cross-validation. The test set concept is represented by the forward-window labels themselves (actuals from 2017).

In [ ]:
# ── Drop non-feature columns ─────────────────────────────────────────────
DROP_COLS = [KEY, TARGET]
# Drop any object columns still remaining (shouldn't be any after nb04 encoding)
object_cols = df.select_dtypes(include='object').columns.tolist()
if object_cols:
    print(f'Dropping remaining object columns: {object_cols}')
    DROP_COLS += object_cols

FEATURE_COLS = [c for c in df.columns if c not in DROP_COLS]
print(f'Feature count: {len(FEATURE_COLS)}')

X = df[FEATURE_COLS].copy()
y = df[TARGET].copy()

# ── Sanity check: no nulls, no inf ───────────────────────────────────────
assert X.isnull().sum().sum() == 0, "Nulls found — fix in notebook 04"
X = X.replace([np.inf, -np.inf], 0)

print(f'X shape: {X.shape}')
print(f'Class balance — churned: {y.sum():,} ({y.mean():.1%})  retained: {(~y.astype(bool)).sum():,}')

---
## 3. Model 1 — Logistic Regression (Interpretable Baseline)
Logistic regression with cross-validation establishes the performance floor. If our complex model doesn't beat this substantially, the behavioural features aren't adding value.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

lr = LogisticRegression(
    C=0.1,               # L2 regularisation — prevents overfitting on correlated features
    class_weight='balanced',  # compensates for churn minority class
    max_iter=1000,
    random_state=42
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

lr_auc_scores = cross_val_score(lr, X_scaled, y, cv=cv, scoring='roc_auc', n_jobs=-1)
lr_ap_scores  = cross_val_score(lr, X_scaled, y, cv=cv, scoring='average_precision', n_jobs=-1)

print('Logistic Regression — 5-fold cross-validation:')
print(f'  ROC-AUC  : {lr_auc_scores.mean():.4f} +/- {lr_auc_scores.std():.4f}')
print(f'  Avg Prec : {lr_ap_scores.mean():.4f}  +/- {lr_ap_scores.std():.4f}')

# Fit on full data for coefficient inspection
lr.fit(X_scaled, y)
joblib.dump(scaler, MODEL / 'scaler.pkl')
print('Scaler saved.')

In [ ]:
# ── Top logistic regression coefficients ─────────────────────────────────
coef_df = pd.DataFrame({
    'feature'    : FEATURE_COLS,
    'coefficient': lr.coef_[0]
}).sort_values('coefficient', key=abs, ascending=False).head(20)

fig, ax = plt.subplots(figsize=(10, 7))
colors = [CHURN_COLOR if v > 0 else RETAIN_COLOR for v in coef_df['coefficient']]
ax.barh(coef_df['feature'], coef_df['coefficient'], color=colors, alpha=0.8, edgecolor='white')
ax.axvline(0, color='gray', linewidth=0.8)
ax.set_xlabel('Coefficient (positive = increases churn risk)')
ax.set_title('Logistic Regression — top 20 feature coefficients')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(FIG / '05_lr_coefficients.png', bbox_inches='tight')
plt.show()

---
## 4. Model 2 — XGBoost (Primary Model)
XGBoost handles mixed feature types, missing values, and non-linear interactions without manual tuning.
`scale_pos_weight` corrects for class imbalance — critical when churn is a minority class.

In [ ]:
# Class imbalance weight
neg = (y == 0).sum()
pos = (y == 1).sum()
scale_pos_weight = neg / pos
print(f'scale_pos_weight: {scale_pos_weight:.2f}  (neg={neg:,} / pos={pos:,})')

xgb_model = xgb.XGBClassifier(
    n_estimators       = 500,
    learning_rate      = 0.05,
    max_depth          = 5,
    subsample          = 0.8,
    colsample_bytree   = 0.8,
    scale_pos_weight   = scale_pos_weight,
    eval_metric        = 'auc',
    use_label_encoder  = False,
    random_state       = 42,
    n_jobs             = -1
)

# Cross-validation
xgb_auc_scores = cross_val_score(xgb_model, X, y, cv=cv, scoring='roc_auc', n_jobs=-1)
xgb_ap_scores  = cross_val_score(xgb_model, X, y, cv=cv, scoring='average_precision', n_jobs=-1)

print('\nXGBoost — 5-fold cross-validation:')
print(f'  ROC-AUC  : {xgb_auc_scores.mean():.4f} +/- {xgb_auc_scores.std():.4f}')
print(f'  Avg Prec : {xgb_ap_scores.mean():.4f}  +/- {xgb_ap_scores.std():.4f}')
print(f'\nImprovement over LR (AUC): +{xgb_auc_scores.mean() - lr_auc_scores.mean():.4f}')

In [ ]:
# ── Fit final model on all data ───────────────────────────────────────────
xgb_model.fit(X, y, verbose=False)
joblib.dump(xgb_model, MODEL / 'churn_model.pkl')
print('XGBoost model saved to models/churn_model.pkl')

# ── XGBoost native feature importance ─────────────────────────────────────
fi = pd.DataFrame({
    'feature'   : FEATURE_COLS,
    'importance': xgb_model.feature_importances_
}).sort_values('importance', ascending=False).head(25)

fig, ax = plt.subplots(figsize=(10, 8))
colors = [
    CHURN_COLOR  if any(k in f for k in ['hyperbolic','recency','loss','aversion','streak','trajectory'])
    else RETAIN_COLOR if any(k in f for k in ['fe_','seasonal'])
    else '#1A8A4A' if any(k in f for k in ['clv','tier','salary'])
    else '#566573'
    for f in fi['feature']
]
ax.barh(fi['feature'], fi['importance'], color=colors, alpha=0.85, edgecolor='white')
ax.set_xlabel('XGBoost feature importance (gain)')
ax.set_title('XGBoost — top 25 features by importance')
ax.invert_yaxis()

from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(facecolor=CHURN_COLOR,  label='Behavioural'),
    Patch(facecolor=RETAIN_COLOR, label='Econometric'),
    Patch(facecolor='#1A8A4A',    label='Demographic/CLV'),
    Patch(facecolor='#566573',    label='Raw activity'),
], fontsize=8)
plt.tight_layout()
plt.savefig(FIG / '05_xgb_feature_importance.png', bbox_inches='tight')
plt.show()

---
## 5. SHAP Explanations — Why Did Each Member Churn?
SHAP (SHapley Additive exPlanations) gives each feature a contribution score for every individual prediction. This powers the Streamlit dashboard's member-level explanation.

In [ ]:
print('Computing SHAP values (may take 1-2 minutes)...')
explainer   = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X)

# Store SHAP values as a DataFrame for the dashboard
shap_df = pd.DataFrame(shap_values, columns=FEATURE_COLS)
shap_df.insert(0, KEY, df[KEY].values)
shap_df.to_csv(PROC / '05_shap_values.csv', index=False)
print('SHAP values saved.')

# Also save the explainer
joblib.dump(explainer, MODEL / 'shap_explainer.pkl')
print('SHAP explainer saved.')

In [ ]:
# ── SHAP summary plot — global feature importance ─────────────────────────
fig, ax = plt.subplots(figsize=(10, 8))
shap.summary_plot(shap_values, X, feature_names=FEATURE_COLS,
                  show=False, plot_size=None)
plt.title('SHAP summary — feature impact on churn prediction', fontsize=13)
plt.tight_layout()
plt.savefig(FIG / '05_shap_summary.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── SHAP bar plot — mean absolute impact ──────────────────────────────────
shap.summary_plot(shap_values, X, feature_names=FEATURE_COLS,
                  plot_type='bar', show=False)
plt.title('SHAP mean |value| — overall feature importance', fontsize=13)
plt.tight_layout()
plt.savefig(FIG / '05_shap_bar.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── SHAP waterfall for a single HIGH-RISK member ────────────────────────
# Find the churned member with the highest predicted probability
df['churn_prob'] = xgb_model.predict_proba(X)[:, 1]

high_risk_idx = df[df[TARGET] == 1]['churn_prob'].idxmax()
member_id     = df.loc[high_risk_idx, KEY]

shap_exp = shap.Explanation(
    values         = shap_values[high_risk_idx],
    base_values    = explainer.expected_value,
    data           = X.iloc[high_risk_idx].values,
    feature_names  = FEATURE_COLS
)
shap.waterfall_plot(shap_exp, max_display=15, show=False)
plt.title(f'SHAP waterfall — high-risk churner (member: {member_id})', fontsize=11)
plt.tight_layout()
plt.savefig(FIG / '05_shap_waterfall_highrisk.png', bbox_inches='tight')
plt.show()

print(f'High-risk member: {member_id}')
print(f'Churn probability: {df.loc[high_risk_idx, "churn_prob"]:.1%}')

In [ ]:
# ── SHAP waterfall for a single LOW-RISK retained member ─────────────────
low_risk_idx = df[df[TARGET] == 0]['churn_prob'].idxmin()
member_id_lr = df.loc[low_risk_idx, KEY]

shap_exp_lr = shap.Explanation(
    values        = shap_values[low_risk_idx],
    base_values   = explainer.expected_value,
    data          = X.iloc[low_risk_idx].values,
    feature_names = FEATURE_COLS
)
shap.waterfall_plot(shap_exp_lr, max_display=15, show=False)
plt.title(f'SHAP waterfall — low-risk retained member (member: {member_id_lr})', fontsize=11)
plt.tight_layout()
plt.savefig(FIG / '05_shap_waterfall_lowrisk.png', bbox_inches='tight')
plt.show()

---
## 6. Model 3 — Cox Proportional Hazard (Econometric Time-to-Churn)
The Cox model answers a different question: **not just WHETHER a member will churn, but WHEN**.
Output is a survival curve per member — gives the marketing team a 30/60/90-day risk timeline.
This is the econometric credibility layer that separates your project from pure ML submissions.

In [ ]:
if LIFELINES_AVAILABLE:
    # Cox needs: duration (tenure), event (churned), and covariates
    # Use a subset of the most important features to keep the model stable
    TOP_COX_FEATURES = [
        'months_since_last_flight', 'hyperbolic_flight_score',
        'recency_ratio', 'loss_aversion_score', 'redemption_ratio',
        'tenure_months', 'flight_trajectory', 'seasonal_volatility',
        'mean_fe_deviation', 'flight_cv'
    ]
    TOP_COX_FEATURES = [f for f in TOP_COX_FEATURES if f in FEATURE_COLS]

    cox_df = X[TOP_COX_FEATURES].copy()
    cox_df['churned']       = y.values
    cox_df['tenure_months'] = cox_df['tenure_months'] if 'tenure_months' in cox_df.columns                               else df.get('tenure_months', pd.Series(12, index=df.index))

    # Clip extreme values to help Cox convergence
    for col in TOP_COX_FEATURES:
        p1, p99 = cox_df[col].quantile(0.01), cox_df[col].quantile(0.99)
        cox_df[col] = cox_df[col].clip(p1, p99)

    # Standardise covariates for Cox (Cox is sensitive to scale)
    from sklearn.preprocessing import StandardScaler as SS
    cov_cols     = [c for c in TOP_COX_FEATURES if c != 'tenure_months']
    cox_scaler   = SS()
    cox_df[cov_cols] = cox_scaler.fit_transform(cox_df[cov_cols])

    cph = CoxPHFitter(penalizer=0.1)  # L2 penalty for stability
    cph.fit(cox_df, duration_col='tenure_months', event_col='churned',
            show_progress=False)

    print('Cox PH model fitted.')
    print(f'Concordance index (C-stat): {cph.concordance_index_:.4f}')
    print('  (0.5 = random, 1.0 = perfect — comparable to AUC)')
    cph.print_summary()
    joblib.dump(cph, MODEL / 'cox_model.pkl')
    print('Cox model saved.')
else:
    print('Skipping Cox — install lifelines first.')
    cph = None

In [ ]:
if LIFELINES_AVAILABLE and cph is not None:
    # ── Cox coefficient plot ──────────────────────────────────────────────
    cox_summary = cph.summary.reset_index()
    cox_summary = cox_summary.sort_values('coef', key=abs, ascending=False)

    fig, ax = plt.subplots(figsize=(9, 6))
    colors = [CHURN_COLOR if v > 0 else RETAIN_COLOR for v in cox_summary['coef']]
    ax.barh(cox_summary['covariate'], cox_summary['coef'],
            xerr=cox_summary['se(coef)'], color=colors, alpha=0.8,
            edgecolor='white', capsize=3)
    ax.axvline(0, color='gray', linewidth=0.8, linestyle='--')
    ax.set_xlabel('Cox coefficient (log hazard ratio)')
    ax.set_title('Cox PH model — hazard ratio coefficients with 95% CI')
    ax.invert_yaxis()
    plt.tight_layout()
    plt.savefig(FIG / '05_cox_coefficients.png', bbox_inches='tight')
    plt.show()

    # ── Survival curves for 3 member archetypes ───────────────────────────
    # High-risk, medium-risk, low-risk
    df['churn_prob_rank'] = df['churn_prob'].rank(pct=True)
    idx_high = df[(df['churn_prob_rank'] > 0.9)  & (df[TARGET]==1)].index[0]
    idx_mid  = df[(df['churn_prob_rank'].between(0.45, 0.55))].index[0]
    idx_low  = df[(df['churn_prob_rank'] < 0.1)  & (df[TARGET]==0)].index[0]

    fig, ax = plt.subplots(figsize=(10, 5))
    for idx, label, color in [
        (idx_high, 'High-risk (churned)',    CHURN_COLOR),
        (idx_mid,  'Mid-risk',               '#E67E22'),
        (idx_low,  'Low-risk (retained)',    RETAIN_COLOR)
    ]:
        row = cox_df.iloc[idx][cov_cols].to_frame().T
        sf  = cph.predict_survival_function(row)
        ax.plot(sf.index, sf.values.flatten(), color=color, linewidth=2, label=label)

    ax.set_xlabel('Months of membership')
    ax.set_ylabel('Survival probability (still engaged)')
    ax.set_title('Cox survival curves — three member archetypes')
    ax.legend()
    ax.set_ylim(0, 1)
    plt.tight_layout()
    plt.savefig(FIG / '05_cox_survival_curves.png', bbox_inches='tight')
    plt.show()

---
## 7. Model Comparison & Business Threshold Selection
AUC tells you model quality. But for the marketing team, the decision is: **at what probability threshold do we intervene?**
A false negative (missing an at-risk member) costs more than a false positive (sending a retention offer to a loyal member).
We set the threshold to maximise recall for the churned class at acceptable precision.

In [ ]:
from sklearn.metrics import precision_recall_curve, roc_curve

# Predictions from final fitted models
lr_probs  = lr.predict_proba(X_scaled)[:, 1]
xgb_probs = df['churn_prob'].values

# ── ROC curve comparison ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for probs, label, color, ls in [
    (lr_probs,  'Logistic Regression', RETAIN_COLOR, '--'),
    (xgb_probs, 'XGBoost',             CHURN_COLOR,  '-')
]:
    fpr, tpr, _ = roc_curve(y, probs)
    auc_val     = roc_auc_score(y, probs)
    axes[0].plot(fpr, tpr, color=color, lw=2, ls=ls,
                 label=f'{label} (AUC={auc_val:.3f})')

axes[0].plot([0,1],[0,1], 'k--', lw=0.8, alpha=0.4)
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC curve comparison')
axes[0].legend()

# ── Precision-Recall curve comparison ────────────────────────────────────
for probs, label, color, ls in [
    (lr_probs,  'Logistic Regression', RETAIN_COLOR, '--'),
    (xgb_probs, 'XGBoost',             CHURN_COLOR,  '-')
]:
    prec, rec, _ = precision_recall_curve(y, probs)
    ap_val       = average_precision_score(y, probs)
    axes[1].plot(rec, prec, color=color, lw=2, ls=ls,
                 label=f'{label} (AP={ap_val:.3f})')

axes[1].axhline(y.mean(), color='gray', linestyle=':', lw=1,
                label=f'Baseline (prevalence={y.mean():.2f})')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall curve comparison')
axes[1].legend()

plt.suptitle('Model comparison: Logistic Regression vs XGBoost', fontsize=13)
plt.tight_layout()
plt.savefig(FIG / '05_model_comparison_curves.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Threshold selection for XGBoost ──────────────────────────────────────
# Business logic: we want recall >= 0.70 (catch 70% of churners)
# Find the threshold that achieves this with maximum precision

prec_arr, rec_arr, thresh_arr = precision_recall_curve(y, xgb_probs)

# Find threshold where recall >= 0.70
valid_mask   = rec_arr >= 0.70
if valid_mask.any():
    best_idx   = np.argmax(prec_arr[valid_mask])
    best_thresh = thresh_arr[valid_mask][best_idx] if best_idx < len(thresh_arr) else 0.5
    best_prec   = prec_arr[valid_mask][best_idx]
    best_rec    = rec_arr[valid_mask][best_idx]
else:
    best_thresh = 0.5
    best_prec   = 0
    best_rec    = 0

print(f'Optimal business threshold: {best_thresh:.3f}')
print(f'At this threshold:')
print(f'  Precision : {best_prec:.3f}  (of members we flag, this % are truly at risk)')
print(f'  Recall    : {best_rec:.3f}  (of all churners, we catch this %)')

# Apply threshold
y_pred_optimal = (xgb_probs >= best_thresh).astype(int)
print('\nClassification report at optimal threshold:')
print(classification_report(y, y_pred_optimal, target_names=['Retained','Churned']))

In [ ]:
# ── Confusion matrix ──────────────────────────────────────────────────────
cm = confusion_matrix(y, y_pred_optimal)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt=',', cmap='Blues',
            xticklabels=['Pred: Retained','Pred: Churned'],
            yticklabels=['True: Retained','True: Churned'],
            ax=ax, cbar=False)
ax.set_title(f'Confusion matrix (threshold={best_thresh:.3f})')
plt.tight_layout()
plt.savefig(FIG / '05_confusion_matrix.png', bbox_inches='tight')
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f'\nBusiness interpretation:')
print(f'  True Positives  (churners caught)       : {tp:,}')
print(f'  False Negatives (churners missed)       : {fn:,}  <- keep this low')
print(f'  False Positives (loyal members flagged) : {fp:,}  <- acceptable cost')
print(f'  True Negatives  (correctly ignored)     : {tn:,}')

---
## 8. Churn Risk Buckets
Rather than a binary flag, give the marketing team a 4-tier risk system they can act on differently.

In [ ]:
def assign_risk_bucket(prob):
    if prob >= 0.75:   return 'Critical (>75%)'
    elif prob >= 0.50: return 'High (50-75%)'
    elif prob >= 0.25: return 'Medium (25-50%)'
    else:              return 'Low (<25%)'

df['risk_bucket'] = df['churn_prob'].apply(assign_risk_bucket)
bucket_order = ['Critical (>75%)','High (50-75%)','Medium (25-50%)','Low (<25%)']

bucket_summary = (
    df.groupby('risk_bucket')
    .agg(
        n_members      = (KEY,     'count'),
        actual_churn   = (TARGET,  'sum'),
        churn_rate     = (TARGET,  'mean'),
        avg_churn_prob = ('churn_prob', 'mean')
    )
    .reindex(bucket_order)
    .reset_index()
)
bucket_summary['pct_of_total'] = bucket_summary['n_members'] / len(df)

print('Risk bucket summary:')
display(bucket_summary.round(3))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors_b = [CHURN_COLOR, '#E67E22', '#F1C40F', RETAIN_COLOR]

bars = axes[0].bar(bucket_summary['risk_bucket'], bucket_summary['n_members'],
                   color=colors_b, alpha=0.85, edgecolor='white')
for bar, row in zip(bars, bucket_summary.itertuples()):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+20,
                 f'{row.n_members:,}\n({row.pct_of_total:.1%})',
                 ha='center', fontsize=9)
axes[0].set_ylabel('Members')
axes[0].set_title('Member count per risk bucket')
axes[0].tick_params(axis='x', rotation=15)

axes[1].bar(bucket_summary['risk_bucket'], bucket_summary['churn_rate'],
            color=colors_b, alpha=0.85, edgecolor='white')
axes[1].set_ylabel('Actual churn rate')
axes[1].set_title('Actual churn rate per risk bucket')
axes[1].yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
axes[1].tick_params(axis='x', rotation=15)

plt.suptitle('Churn risk bucket distribution', fontsize=13)
plt.tight_layout()
plt.savefig(FIG / '05_risk_buckets.png', bbox_inches='tight')
plt.show()

---
## 9. Churn Probability Distribution — Model Calibration Check

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution of predicted probabilities
for churn_val, label, color in [(0,'Retained',RETAIN_COLOR),(1,'Churned',CHURN_COLOR)]:
    sub = df[df[TARGET]==churn_val]['churn_prob']
    axes[0].hist(sub, bins=50, alpha=0.55, color=color,
                 label=label, density=True, edgecolor='white')
axes[0].axvline(best_thresh, color='black', linestyle='--',
                linewidth=1.5, label=f'Threshold ({best_thresh:.2f})')
axes[0].set_xlabel('Predicted churn probability')
axes[0].set_ylabel('Density')
axes[0].set_title('Predicted probability distribution by true label')
axes[0].legend()

# Calibration: predicted vs actual by probability decile
df['prob_decile'] = pd.qcut(df['churn_prob'], q=10, labels=False, duplicates='drop')
calibration = df.groupby('prob_decile').agg(
    mean_predicted = ('churn_prob', 'mean'),
    actual_rate    = (TARGET, 'mean')
).reset_index()

axes[1].scatter(calibration['mean_predicted'], calibration['actual_rate'],
                color=CHURN_COLOR, s=80, zorder=3)
axes[1].plot([0,1],[0,1], 'k--', lw=1, alpha=0.5, label='Perfect calibration')
axes[1].set_xlabel('Mean predicted probability (decile)')
axes[1].set_ylabel('Actual churn rate (decile)')
axes[1].set_title('Calibration plot — predicted vs actual by decile')
axes[1].legend()

plt.suptitle('XGBoost model calibration', fontsize=13)
plt.tight_layout()
plt.savefig(FIG / '05_calibration.png', bbox_inches='tight')
plt.show()

---
## 10. Save Predictions & Model Metadata

In [ ]:
# Full prediction table — this powers the Streamlit dashboard
predictions = df[[KEY, TARGET, 'churn_prob', 'risk_bucket']].copy()
predictions['churn_pct']      = (predictions['churn_prob'] * 100).round(1)
predictions['predicted_label'] = (predictions['churn_prob'] >= best_thresh).astype(int)

predictions.to_csv(PROC / '05_predictions.csv', index=False)
print(f'Saved: 05_predictions.csv  ({len(predictions):,} members)')

# Also save feature column list — Streamlit needs this for inference
import json
model_meta = {
    'feature_cols'   : FEATURE_COLS,
    'threshold'      : round(float(best_thresh), 4),
    'xgb_cv_auc'     : round(float(xgb_auc_scores.mean()), 4),
    'lr_cv_auc'      : round(float(lr_auc_scores.mean()), 4),
    'churn_rate'     : round(float(y.mean()), 4),
    'n_members'      : int(len(df)),
    'risk_buckets'   : bucket_order
}
with open(MODEL / 'model_meta.json', 'w') as f:
    json.dump(model_meta, f, indent=2)
print('Saved: models/model_meta.json')

---
## 11. Summary — Numbers for Your Report

In [ ]:
print('='*60)
print('  MODELLING COMPLETE — KEY NUMBERS FOR YOUR REPORT')
print('='*60)
print(f'  Logistic Regression CV AUC  : {lr_auc_scores.mean():.4f}')
print(f'  XGBoost CV AUC              : {xgb_auc_scores.mean():.4f}')
if LIFELINES_AVAILABLE and cph is not None:
    print(f'  Cox C-statistic             : {cph.concordance_index_:.4f}')
print(f'  Chosen threshold            : {best_thresh:.3f}')
print(f'  Precision at threshold      : {best_prec:.3f}')
print(f'  Recall at threshold         : {best_rec:.3f}')
print(f'  Churners caught (TP)        : {tp:,}')
print(f'  Churners missed (FN)        : {fn:,}')
print()
print('  Risk bucket breakdown:')
for _, row in bucket_summary.iterrows():
    print(f'    {row["risk_bucket"]:<22}: {row["n_members"]:>5,} members  '
          f'actual churn rate: {row["churn_rate"]:.1%}')
print('='*60)
print('\nAll models saved to models/')
print('Predictions saved to data/processed/05_predictions.csv')
print('Next step -> 06_customer_segmentation.ipynb')